# Pareja C · Idea 6 — Apilamiento (stacking)

Una etapa 2 recibe las 17 probabilidades y las corrige con cómo se juntan los géneros (Misterio casi siempre va con Thriller o Crimen). Se entrena con predicciones **fuera de pliegue** (5 pliegues de train, por franquicia) para no sobreajustar. Empezar lineal 17 → 17 con `residual: True` (solo ajusta); luego MLP si ayuda. El formato del `.pt` ya existe (contrato 4).

**Dónde se programa:** `apilamiento.py` (`pliegues`, `apilar`). La lógica va en los `.py` (se revisa en el Pull Request); este
notebook solo la corre. **Listo cuando** su prueba pasa (quiten el `xfail` de `test_idea6` en
`tests/test_pendientes.py`) y el resultado en validación está en el registro.

Antes, una vez para todo el equipo: `00_reproducir_base` (la base `v0.1.0`).

## Preparar el entorno

**CPU basta**: entrenar una cabeza con vectores ya calculados tarda segundos.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`C-apilamiento`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "C-apilamiento"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt

In [ ]:
# ¿ya pasa la prueba de la idea? (mientras esté pendiente sale como "xfailed")
!python -m pytest -q tests/test_pendientes.py -k idea6 -rxX

## 1. Apilar sobre la base (después, sobre el mejor modelo o el ensamble)

In [ ]:
!python -m clasificador.apilamiento C6_apilamiento base
!python -m clasificador.evaluar C6_apilamiento
!python -m clasificador.comparar base C6_apilamiento

## 2. Mapa de calor 17 × 17 (figura para el reporte): qué género empuja a cuál

In [ ]:
import matplotlib.pyplot as plt, os, torch
from p2m.config import GENEROS_ES
guardado = torch.load(os.environ["P2M_SALIDAS"] + "/modelos/C6_apilamiento.pt", weights_only=False)
pesos = guardado["apilamiento"]["capa"]["weight"].numpy()   # etapa 2 lineal: (salida, entrada)
etiquetas = [GENEROS_ES[g] for g in guardado["generos"]]
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(pesos, cmap="RdBu_r", vmin=-abs(pesos).max(), vmax=abs(pesos).max())
ax.set_xticks(range(17), etiquetas, rotation=90); ax.set_yticks(range(17), etiquetas)
ax.set_xlabel("probabilidad que entra"); ax.set_ylabel("género corregido"); fig.colorbar(im)
os.makedirs("resultados/C6_apilamiento", exist_ok=True)
fig.savefig("resultados/C6_apilamiento/mapa_calor.png", bbox_inches="tight", dpi=150)

## Subir a la rama `C-apilamiento`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`C-apilamiento`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Fine-grained tokens*).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados clasificador tests
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")